# 06. Atok boundary map

The frozen model from notebook 03 applied to the wider KMZ **Atok** boundary (about 1,789 ha; the Paoay study area is its "Region of Interest" row). Sentinel-2 comes from the local Cloud Score+ composites (most complete year), Sentinel-1 from Earth Engine `computePixels` in row strips (cached under `pipeline_outputs/improved/cache/atok_boundary_<year>/`). The land-activity map covers only Paoay, so cropland here is **ESA WorldCover** cropland (class 40). Unsure pixels become Unknown (0.60 / 0.10 rule). Exploratory: no local labels.

**Input:** `current_models.json`. **Output:** `current_atok.json`, `crop_map.tif`, `probabilities.tif`, `area_summary.csv`. Notebook 04's Paoay map is unchanged.

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from IPython.display import display
from model_aza.improved import workflow
pd.set_option("display.width", 200)

## Map

In [ ]:
models_handoff = ""  # Optional models receipt; blank uses the current pointer
receipt = workflow.atok_map(root, models_handoff or None)
run = receipt.parent
summary = json.loads((run / "paoay_summary.json").read_text())
print("Atok receipt:", receipt)
display(pd.Series({k: v for k, v in summary.items() if not isinstance(v, dict)}).to_frame("value"))
display(pd.DataFrame({"map (confident cropland)": summary["crop_shares_of_confident_cropland"],
                      "Benguet 2024": summary["benguet_2024_shares"]}).round(3))
display(pd.read_csv(run / "area_summary.csv"))

## Preview

In [ ]:
import rasterio
with rasterio.open(run / "crop_map.tif") as raster:
    codes, cropland = raster.read(1), raster.read(2)
colors = ListedColormap(["#ffffff", "#ff8c00", "#9acd60", "#7a5444", "#c8c8c8", "#eeeeee"])
figure, axes = plt.subplots(1, 2, figsize=(14, 6.5))
axes[0].imshow(codes, cmap=colors, vmin=0, vmax=5, interpolation="nearest")
axes[0].set_title("Crop map: carrot (orange), cabbage (green), potato (brown), Unknown (grey)", fontsize=9)
axes[1].imshow(cropland, cmap=ListedColormap(["#ffffff", "#eeeeee", "#1baf7a"]), vmin=0, vmax=2, interpolation="nearest")
axes[1].set_title("ESA WorldCover cropland (green)", fontsize=9)
for axis in axes:
    axis.axis("off")
figure.suptitle(f"Atok boundary, {summary['year']} (exploratory, not validated)")
figure.tight_layout()
figure.savefig(run / "preview.png", dpi=150)
display(figure)
plt.close(figure)

## Sentinel-1 coverage

In [ ]:
cache = root / "pipeline_outputs/improved/cache" / f"atok_boundary_{summary['year']}" / f"s1_grid_{summary['year']}.npy"
s1 = np.load(cache)
present = np.isfinite(s1).all(axis=1)  # (24 bins, rows, cols)
inside = codes > 0
print("Sentinel-1 cube:", s1.shape, "| half-months with both polarizations, mean over boundary pixels:",
      round(float(present[:, inside].sum(axis=0).mean()), 1), "of 24")